In [ ]:
!pip -q install -U "autogen-agentchat" "autogen-ext[openai]" cohere

In [2]:
!pip install -U autogen-agentchat "autogen-ext[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.26.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.41.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatib

In [1]:

import os
from getpass import getpass

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

# 1. Enter Cohere API key
api_key = getpass("Enter Cohere API key: ")

# 2. Connect AutoGen to Cohere's compatibility endpoint
model_client = OpenAIChatCompletionClient(
    model="command-a-03-2025",
    base_url="https://api.cohere.com/compatibility/v1",
    api_key=api_key,
    model_info=ModelInfo(
        family="unknown",
        vision=False,
        function_calling=True,
        json_output=False,
        structured_output=False
    )
)

# 3. Define tools the agent can choose to use
knowledge_base = {
    "machine learning": (
        "Machine learning enables systems to learn patterns "
        "from data. Main types include supervised, "
        "unsupervised, and reinforcement learning."
    ),
    "deep learning": (
        "Deep learning uses multilayer neural networks. "
        "Applications include computer vision, speech, "
        "and natural language processing."
    ),
    "rag": (
        "Retrieval-Augmented Generation retrieves relevant "
        "documents and supplies them as context to an LLM."
    )
}

def search_knowledge_base(topic: str) -> str:
    """Search the local knowledge base for a topic."""
    matches = [
        f"{key}: {value}"
        for key, value in knowledge_base.items()
        if key in topic.lower() or topic.lower() in key
    ]

    if matches:
        return "\n".join(matches)

    return "No exact topic match found in the local knowledge base."


def calculate_word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())


# 4. Create the autonomous agent
agent = AssistantAgent(
    name="research_agent",
    model_client=model_client,
    tools=[search_knowledge_base, calculate_word_count],
    system_message="""
    You are an autonomous academic research assistant.

    Your objective is to prepare a short research brief
    on the topic provided by the user.

    You may independently decide when to:
    1. Search the local knowledge base.
    2. Calculate the word count of your draft.
    3. Revise the draft if needed.

    Use tools when they help accomplish the task.
    Do not claim to have searched external websites.
    If information is unavailable, state the limitation.
    Produce a final brief with:
    - Topic overview
    - Key concepts
    - Applications
    - Limitations
    """,
    reflect_on_tool_use=True
)

# 5. Run the agent
task = """
Prepare a research brief on machine learning.
Use the knowledge base to ground the explanation.
Keep the final report around 150 words.
"""

result = await agent.run(task=task)

# 6. Display the complete execution history
for message in result.messages:
    print(f"\n--- {message.source} ---")
    print(message.content)

# 7. Close the model connection
await model_client.close()


Enter Cohere API key: ··········

--- user ---

Prepare a research brief on machine learning.
Use the knowledge base to ground the explanation.
Keep the final report around 150 words.


--- research_agent ---
[FunctionCall(id='search_knowledge_base_jy7xfw4wv7r2', arguments='{"topic":"machine learning"}', name='search_knowledge_base')]

--- research_agent ---
[FunctionExecutionResult(content='machine learning: Machine learning enables systems to learn patterns from data. Main types include supervised, unsupervised, and reinforcement learning.', name='search_knowledge_base', call_id='search_knowledge_base_jy7xfw4wv7r2', is_error=False)]

--- research_agent ---
# Machine Learning Research Brief  

**Topic Overview**: Machine learning (ML) is a subset of artificial intelligence that enables systems to learn patterns from data without explicit programming. It focuses on developing algorithms that improve performance on tasks through experience.  

**Key Concepts**: The main types of ML incl

In [ ]:
!pip install -q gradio

import gradio as gr
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

def init_agent():
    # Re-initialize the model client and agent to ensure we have a fresh, non-closed connection
    model_client = OpenAIChatCompletionClient(
        model="command-a-03-2025",
        base_url="https://api.cohere.com/compatibility/v1",
        api_key=api_key,
        model_info=ModelInfo(
            family="unknown",
            vision=False,
            function_calling=True,
            json_output=False,
            structured_output=False
        )
    )

    agent = AssistantAgent(
        name="research_agent",
        model_client=model_client,
        tools=[search_knowledge_base, calculate_word_count],
        system_message="""
        You are an autonomous academic research assistant.

        Your objective is to prepare a short research brief
        on the topic provided by the user.

        You may independently decide when to:
        1. Search the local knowledge base.
        2. Calculate the word count of your draft.
        3. Revise the draft if needed.

        Use tools when they help accomplish the task.
        Do not claim to have searched external websites.
        If information is unavailable, state the limitation.
        Produce a final brief with:
        - Topic overview
        - Key concepts
        - Applications
        - Limitations
        """,
        reflect_on_tool_use=True
    )
    return agent, model_client

async def run_research(task_input):
    agent, model_client = init_agent()
    output_history = []
    try:
        result = await agent.run(task=task_input)
        for message in result.messages:
            content_str = str(message.content)
            output_history.append(f"**{message.source}**:\n{content_str}\n")
    except Exception as e:
        output_history.append(f"An error occurred: {str(e)}")
    finally:
        await model_client.close()
    return "\n---\n".join(output_history)

def sync_run_research(task_input):
    # Helper to run async research function in a synchronous context
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(run_research(task_input))

# Build Gradio UI
with gr.Blocks(title="Research Assistant Agent") as demo:
    gr.Markdown("# 🤖 AutoGen Research Assistant")
    gr.Markdown("Enter a research query, and the agent will use its available tools to research and produce a brief.")

    with gr.Row():
        with gr.Column():
            task_input = gr.Textbox(
                label="Research Query / Task",
                placeholder="e.g., Prepare a research brief on machine learning. Keep it around 150 words.",
                lines=4
            )
            submit_btn = gr.Button("Run Agent", variant="primary")
        with gr.Column():
            output_display = gr.Markdown(label="Agent Execution Log")

    submit_btn.click(
        fn=sync_run_research,
        inputs=task_input,
        outputs=output_display
    )

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://79a482238c76edeaf6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
